In [ ]:
import sys
sys.path.append("..")
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from src.utils.config_loader import load_config

cfg = load_config("../config.yaml")

In [ ]:
student_model_id = cfg.model_student.model_id
max_seq_length = cfg.model_student.max_seq_length
load_in_4bit = cfg.model_student.load_in_4bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=student_model_id,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# Unsloth 需要 specific 的格式，我们这里构造 ChatML 格式
rest_dataset = load_dataset("json", data_files="../data/processed/train_rest_cot_distilled.jsonl", split="train")
lap_dataset = load_dataset("json", data_files="../data/processed/train_lap_cot_distilled.jsonl", split="train")

def format_for_sft(example):
    # Student 的输入不包含 label，只有 text 和 aspect
    user_content = f"Analyze sentiment. Sentence: {example['text']}\nAspect: {example['aspect']}"
    
    return {
        "input": user_content,
        "label": example["student_response"]
    }

rest_dataset = rest_dataset.map(format_for_sft)
lap_dataset = lap_dataset.map(format_for_sft)
combined_dataset = rest_dataset.concatenate(lap_dataset)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=combined_dataset,
    dataset_text_field="messages", # Unsloth 会自动处理 Chat 格式
    max_seq_length=2048,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=300, # 根据数据量调整 epoch
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="../outputs/qwen3_cot_finetuned",
        optim="adamw_8bit",
    ),
)

trainer.train()

In [ ]:
model.save_pretrained("../outputs/qwen3_cot_finetuned")
tokenizer.save_pretrained("../outputs/qwen3_cot_finetuned")